# 03 · Extraction and verification (the Extractor agent and the NLI Verifier)

**Purpose.** Turn papers into `PaperRecord`s whose every claim quotes its evidence verbatim, then verify each claim twice: a fuzzy check that the quote really occurs in the source, and an NLI entailment check that the claim follows from the quote. The two thresholds are fitted on a hand-checked dev set, not guessed.

| | |
|---|---|
| **Inputs** | `data/papers_train.jsonl` + `data/sources/` (notebook 01) |
| **Outputs** | `data/extractions_*.jsonl`, `data/claim_verdicts_*.json`, `results/verifier_calibration.json`, `results/stage3_*.json` |
| **Expected runtime (Colab T4)** | 1-3 h for 100-300 papers (API-bound); the NLI model loads in ~1 min |
| **Required secrets** | `GROQ_API_KEY` (or `GEMINI_API_KEY`) |
| **GPU** | T4 recommended (the DeBERTa NLI model runs ~10x faster than on CPU) |

Everything is cached under `/content/drive/MyDrive/litreview/` and every stage is
**resumable**: if Colab disconnects, re-run all cells and it continues where it
stopped. Only the cell marked CONFIG needs editing.


In [ ]:
# --- Colab setup: Drive, project code, secrets. Safe to re-run. ---
import json, os, subprocess, sys, zipfile
from pathlib import Path
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

ROOT = "/content/drive/MyDrive/litreview" if ON_COLAB else os.path.abspath("..")
os.makedirs(ROOT, exist_ok=True)
os.environ["LITREVIEW_ROOT"] = ROOT
REPO = os.path.join(ROOT, "repo")

# Optional: a git URL to clone the project from. Leave blank to use the folder or
# the zip on Drive instead.
REPO_URL = ""

def _find_project() -> str | None:
    """Locate the project code: an existing folder, a zip on Drive, or a clone."""
    if os.path.exists(os.path.join(REPO, "pyproject.toml")):
        return REPO
    # a litreview_*.zip uploaded anywhere under MyDrive/litreview
    zips = sorted(Path(ROOT).glob("litreview*.zip")) + sorted(Path(ROOT).glob("**/litreview*.zip"))
    for z in zips:
        print(f"extracting {z.name} ...")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(ROOT)
        # the zip may contain a top-level folder; find the pyproject inside
        for cand in [REPO, ROOT] + [str(p.parent) for p in Path(ROOT).glob("*/pyproject.toml")]:
            if os.path.exists(os.path.join(cand, "pyproject.toml")):
                return cand
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=False)
        if os.path.exists(os.path.join(REPO, "pyproject.toml")):
            return REPO
    return None

if ON_COLAB:
    found = _find_project()
    if not found:
        raise SystemExit(
            "Could not find the project code. Do one of these, then re-run this cell:
"
            "  1. upload litreview_colab.zip into MyDrive/litreview/ (easiest), or
"
            "  2. copy the project folder to MyDrive/litreview/repo, or
"
            "  3. set REPO_URL above to a git URL.
"
            "The notebooks deliberately contain no second copy of the code.")
    REPO = found
    print("project code:", REPO)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", f"{REPO}[ml,eval]"],
                   check=True)

import litreview
from litreview.config import SETTINGS, paths, secret
P = paths().mkdirs()
print("data root:", P.root)
print("already present:", {f.name: f.stat().st_size // 1024 for f in sorted(P.data.glob("*.jsonl"))})


In [ ]:
# ======================= CONFIG (the only cell to edit) =======================
PROVIDER  = "groq"
MODEL     = ""
N_PAPERS  = 100              # papers to extract this session (resumable)
SPLIT     = "train"
BACKEND   = "api"            # "api" = direct LLM call; "crew" = CrewAI agent + output_pydantic
N_DEV     = 60               # claim/evidence pairs to hand-check for calibration
SEED      = 0
# ==============================================================================


In [ ]:
# Provider comes from one config. Keys are read from Colab secrets, never hard-coded.
import os
os.environ["LITREVIEW_PROVIDER"] = PROVIDER
if MODEL:
    os.environ["LITREVIEW_MODEL"] = MODEL

from litreview.config import Settings
from litreview.llm import install_call_logger, probe, usage_summary
import litreview.config as _cfg
_cfg.SETTINGS = Settings()                      # re-read after setting the env vars
import litreview.llm as _llm
_llm.SETTINGS = _cfg.SETTINGS

install_call_logger(P.results / "llm_calls.jsonl")
print(json.dumps(_cfg.SETTINGS.describe(), indent=1))
print(json.dumps(probe(_cfg.SETTINGS), indent=1))


### The schemas

`PaperRecord` and `ReviewSentence` are the contract between every stage, and they
are what the distilled model in notebook 05 has to emit.

In [ ]:
from litreview.schemas import PaperRecord, ReviewSentence
print(json.dumps(PaperRecord.model_json_schema()["properties"], indent=1)[:1500])


### Extract

The Extractor's input is the abstract plus whatever introduction, results,
conclusion and limitations text the LaTeX source gave us, under a character
budget. The exact input string is cached next to the record, because the
Verifier's quote check is defined against *that* string.

`BACKEND = "crew"` runs the same prompt as a CrewAI agent with
`output_pydantic=PaperRecord`; `"api"` calls the model directly and is what the
batch runs use.

In [ ]:
import sys, subprocess
from tqdm.auto import tqdm
from litreview.extract import ApiExtractor, CrewExtractor, extract_corpus, load_records
SCRIPTS = (Path(REPO) / "scripts") if ON_COLAB else Path("../scripts")
sys.path.insert(0, str(SCRIPTS))
from run_stage3 import papers_from_mining       # same helper the CLI uses

papers = papers_from_mining(N_PAPERS, SPLIT, SEED)
print(f"{len(papers)} papers with usable sections")

ex = CrewExtractor() if BACKEND == "crew" else ApiExtractor()
cache = P.data / f"extractions_{SPLIT}_{ex.name}.jsonl"
stats = extract_corpus(papers, cache, extractor=ex,
                       progress=lambda ps: tqdm(ps, desc="extract"))
stats |= {"backend": ex.name, "cache": str(cache), "usage": usage_summary()}
(P.results / f"stage3_extract_{SPLIT}_{ex.name}.json").write_text(json.dumps(stats, indent=2))
print(json.dumps({k: v for k, v in stats.items() if k != "usage"}, indent=2))


In [ ]:
records = load_records(cache)
pid = next(iter(records))
rec = records[pid]["record"]
print(rec["title"][:90], "\n  task:", rec.get("task"))
for c in rec["claims"][:3]:
    print(f"  [{c['claim_id']}] {c['text']}\n      evidence ({c['evidence']['section']}): "
          f"\"{c['evidence']['passage'][:160]}\"")


### Calibrate the Verifier

Sample claim/evidence pairs, label each `supported`, `partially` or `unsupported`
by hand, then fit the quote and entailment thresholds by grid search on macro-F1
over the three verdicts. Macro-F1 rather than accuracy, so a threshold pair is not
chosen for doing well on the majority verdict alone.

In [ ]:
subprocess.run([sys.executable, str(SCRIPTS / "run_stage3.py"), "make-dev",
                "--n", str(N_DEV), "--split", SPLIT], check=False)
print("\nHand-label", P.data / "verifier_dev.jsonl",
      '- set "human" to supported | partially | unsupported for each item.')


In [ ]:
# Run once verifier_dev.jsonl is labelled. Until then the defaults in
# litreview.verify.DEFAULT_THRESHOLDS are used, and the report says they are defaults.
if (P.data / "verifier_dev.jsonl").exists():
    subprocess.run([sys.executable, str(SCRIPTS / "run_stage3.py"), "calibrate"], check=False)
if (P.results / "verifier_calibration.json").exists():
    cal = json.loads((P.results / "verifier_calibration.json").read_text())
    print(json.dumps({k: v for k, v in cal.items() if k not in ("grid", "per_item")}, indent=2))


### Verify every claim

Two independent checks per claim. A claim can quote correctly and still overstate
what the passage says; the entailment check is what catches that.

In [ ]:
from litreview.verify import Verifier
v = Verifier()
print("thresholds:", v.th, "\nNLI model:", v.nli.model_name, "on", v.nli.device)

rep = v.verify_corpus(records, P.data / f"claim_verdicts_{SPLIT}_{ex.name}.json")
(P.results / f"stage3_verify_{SPLIT}_{ex.name}.json").write_text(json.dumps(rep, indent=2))
print(json.dumps(rep, indent=2))


### Next

**04_synthesis_and_gaps.ipynb** — the consensus/contradiction map, gap detection
and the full CrewAI Flow on one real topic.